In [31]:
import pandas as pd

columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

train_df = pd.read_csv("KDDTrain+.txt", names=columns)
test_df = pd.read_csv("KDDTest+.txt", names=columns)

print(train_df.shape)
train_df.head()

(125973, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [32]:
print(train_df.shape)
print(train_df['label'].value_counts())
train_df['binary_label'] = train_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
test_df['binary_label'] = test_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

print(train_df['binary_label'].value_counts())

# Check for missing values
print(train_df.isnull().sum().sum())  # total missing values across all columns

# Check categorical columns
print(train_df['protocol_type'].unique())
print(train_df['service'].unique())
print(train_df['flag'].unique())

from sklearn.preprocessing import LabelEncoder

categorical_cols = ['protocol_type', 'service', 'flag']

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

train_df.head()

# Encode the binary label too (normal=0, attack=1)
from sklearn.preprocessing import LabelEncoder

le_label = LabelEncoder()
train_df['binary_label_encoded'] = le_label.fit_transform(train_df['binary_label'])
test_df['binary_label_encoded'] = le_label.transform(test_df['binary_label'])

# Define features (X) and target (y)
drop_cols = ['label', 'binary_label', 'binary_label_encoded', 'difficulty']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['binary_label_encoded']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['binary_label_encoded']

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(le_label.classes_)  # shows which number = normal, which = attack

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train a simple Decision Tree first (easiest to understand)
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Predict on test data
y_pred = dt_model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['attack', 'normal']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


(125973, 43)
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64
binary_label
normal    67343
attack    58630
Name: count, dtype: int64
0
<ArrowStringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str
<ArrowStringArray>
[   'ftp_data',       'other',     'private',        'http',  'remote_job',
        'name',  'netbios_ns',       'eco_i',         'mtp',      'telnet',
      'finger',    'domain_u',      'supdup',   'uucp_path',      'Z39_5

In [33]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf, target_names=['attack', 'normal']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.772666784953868

Classification Report:
               precision    recall  f1-score   support

      attack       0.97      0.62      0.76     12833
      normal       0.66      0.97      0.79      9711

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544


Confusion Matrix:
 [[7987 4846]
 [ 279 9432]]


In [34]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf, target_names=['attack', 'normal']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.772666784953868

Classification Report:
               precision    recall  f1-score   support

      attack       0.97      0.62      0.76     12833
      normal       0.66      0.97      0.79      9711

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544


Confusion Matrix:
 [[7987 4846]
 [ 279 9432]]


In [35]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf, target_names=['attack', 'normal']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.772666784953868

Classification Report:
               precision    recall  f1-score   support

      attack       0.97      0.62      0.76     12833
      normal       0.66      0.97      0.79      9711

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544


Confusion Matrix:
 [[7987 4846]
 [ 279 9432]]


In [36]:
import numpy as np

# Check if the actual predictions are identical
print("Are predictions identical?", np.array_equal(y_pred, y_pred_rf))

# Check how many predictions differ
print("Number of differing predictions:", np.sum(y_pred != y_pred_rf))

# Sanity check: is rf_model actually a RandomForest with 100 trees?
print("Number of trees in RF:", len(rf_model.estimators_))

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Difference:", accuracy_score(y_test, y_pred_rf) - accuracy_score(y_test, y_pred))

Are predictions identical? False
Number of differing predictions: 1346
Number of trees in RF: 100
Decision Tree Accuracy: 0.789611426543648
Random Forest Accuracy: 0.772666784953868
Difference: -0.01694464158978004


In [37]:
from sklearn.metrics import recall_score

dt_recall_attack = recall_score(y_test, y_pred, pos_label=0)  # 0 = attack
rf_recall_attack = recall_score(y_test, y_pred_rf, pos_label=0)

print("Decision Tree Recall (attack):", dt_recall_attack)
print("Random Forest Recall (attack):", rf_recall_attack)

Decision Tree Recall (attack): 0.6530039741291982
Random Forest Recall (attack): 0.6223798020727811


In [38]:
rf_model_tuned = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42
)
rf_model_tuned.fit(X_train, y_train)
y_pred_rf_tuned = rf_model_tuned.predict(X_test)

print("Tuned RF Accuracy:", accuracy_score(y_test, y_pred_rf_tuned))
print("Tuned RF Recall (attack):", recall_score(y_test, y_pred_rf_tuned, pos_label=0))

Tuned RF Accuracy: 0.7702714691270405
Tuned RF Recall (attack): 0.6170030390399751


In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Logistic Regression Recall (attack):", recall_score(y_test, y_pred_lr, pos_label=0))

Logistic Regression Accuracy: 0.7539478353442157
Logistic Regression Recall (attack): 0.6177043559572976


In [40]:
import pandas as pd

importances = pd.Series(dt_model.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(10)
print(top_features)

src_bytes                      0.754971
protocol_type                  0.066163
dst_host_srv_count             0.050945
dst_host_srv_rerror_rate       0.025014
dst_bytes                      0.021743
hot                            0.021559
service                        0.012337
logged_in                      0.010038
dst_host_same_src_port_rate    0.007685
dst_host_same_srv_rate         0.007447
dtype: float64


In [41]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Gradient Boosting Recall (attack):", recall_score(y_test, y_pred_gb, pos_label=0))

Gradient Boosting Accuracy: 0.7861071682044003
Gradient Boosting Recall (attack): 0.6466141977713706


In [42]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost Recall (attack):", recall_score(y_test, y_pred_xgb, pos_label=0))

XGBoost Accuracy: 0.8047817601135557
XGBoost Recall (attack): 0.6789527000701316


In [44]:
# Re-load fresh data since we'll re-encode differently
train_df2 = pd.read_csv("KDDTrain+.txt", names=columns)
test_df2 = pd.read_csv("KDDTest+.txt", names=columns)

train_df2['binary_label'] = train_df2['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df2['binary_label'] = test_df2['label'].apply(lambda x: 0 if x == 'normal' else 1)

# One-hot encode categorical columns
train_df2 = pd.get_dummies(train_df2, columns=['protocol_type', 'service', 'flag'])
test_df2 = pd.get_dummies(test_df2, columns=['protocol_type', 'service', 'flag'])

# Align columns (test set might be missing some service categories that exist in train)
train_df2, test_df2 = train_df2.align(test_df2, join='left', axis=1, fill_value=0)

X_train2 = train_df2.drop(columns=['label', 'binary_label', 'difficulty'])
y_train2 = train_df2['binary_label']
X_test2 = test_df2.drop(columns=['label', 'binary_label', 'difficulty'])
y_test2 = test_df2['binary_label']

print(X_train2.shape, X_test2.shape)

(125973, 122) (22544, 122)


In [45]:
xgb_model2 = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb_model2.fit(X_train2, y_train2)
y_pred_xgb2 = xgb_model2.predict(X_test2)

print("XGBoost v2 Accuracy:", accuracy_score(y_test2, y_pred_xgb2))
print("XGBoost v2 Recall (attack):", recall_score(y_test2, y_pred_xgb2, pos_label=0))

XGBoost v2 Accuracy: 0.7933374733853797
XGBoost v2 Recall (attack): 0.9718875502008032


In [46]:
print("XGBoost v2 Recall (attack):", recall_score(y_test2, y_pred_xgb2, pos_label=1))

XGBoost v2 Recall (attack): 0.6582248889581548


In [47]:
full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

print("Combined shape:", full_df.shape)
print(full_df['binary_label'].value_counts())

Combined shape: (148517, 45)
binary_label
normal    77054
attack    71463
Name: count, dtype: int64


In [48]:
drop_cols = ['label', 'binary_label', 'binary_label_encoded', 'difficulty']
X_full = full_df.drop(columns=drop_cols)
y_full = full_df['binary_label_encoded']

In [49]:
from sklearn.model_selection import train_test_split

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print("New train shape:", X_train_new.shape)
print("New test shape:", X_test_new.shape)

New train shape: (118813, 41)
New test shape: (29704, 41)


In [50]:
dt_model_new = DecisionTreeClassifier(random_state=42)
dt_model_new.fit(X_train_new, y_train_new)
y_pred_dt_new = dt_model_new.predict(X_test_new)

print("New Decision Tree Accuracy:", accuracy_score(y_test_new, y_pred_dt_new))
print("New Decision Tree Recall (attack):", recall_score(y_test_new, y_pred_dt_new, pos_label=0))

New Decision Tree Accuracy: 0.9942431995690816
New Decision Tree Recall (attack): 0.9944028545441824


In [51]:
xgb_model_new = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model_new.fit(X_train_new, y_train_new)
y_pred_xgb_new = xgb_model_new.predict(X_test_new)

print("New XGBoost Accuracy:", accuracy_score(y_test_new, y_pred_xgb_new))
print("New XGBoost Recall (attack):", recall_score(y_test_new, y_pred_xgb_new, pos_label=0))

New XGBoost Accuracy: 0.9959264745488823
New XGBoost Recall (attack): 0.9950325334079619


In [52]:
# Check correlation of each feature with the target
correlations = X_full.corrwith(y_full).abs().sort_values(ascending=False)
print(correlations.head(10))

same_srv_rate               0.708911
dst_host_srv_count          0.692577
dst_host_same_srv_rate      0.667624
logged_in                   0.664117
flag                        0.629556
dst_host_srv_serror_rate    0.593690
dst_host_serror_rate        0.589936
serror_rate                 0.588474
srv_serror_rate             0.586636
count                       0.524108
dtype: float64


C:\Users\nandhan\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\nandhan\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [53]:
xgb_model_v3 = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb_model_v3.fit(X_train_new, y_train_new)
y_pred_v3 = xgb_model_v3.predict(X_test_new)

print("XGBoost v3 Accuracy:", accuracy_score(y_test_new, y_pred_v3))
print("XGBoost v3 Recall (attack):", recall_score(y_test_new, y_pred_v3, pos_label=0))

XGBoost v3 Accuracy: 0.9966671155399947
XGBoost v3 Recall (attack): 0.9960819981809277


In [54]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:\n", confusion_matrix(y_test_new, y_pred_xgb_new))
print("\nFull Classification Report:\n", classification_report(y_test_new, y_pred_xgb_new, target_names=['attack', 'normal']))

Confusion Matrix:
 [[14222    71]
 [   50 15361]]

Full Classification Report:
               precision    recall  f1-score   support

      attack       1.00      1.00      1.00     14293
      normal       1.00      1.00      1.00     15411

    accuracy                           1.00     29704
   macro avg       1.00      1.00      1.00     29704
weighted avg       1.00      1.00      1.00     29704



In [55]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(xgb_model_new, X_full, y_full, cv=5, scoring='accuracy')
print("Cross-validation scores (5 folds):", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())
print("Std deviation:", cv_scores.std())

Cross-validation scores (5 folds): [0.99868705 0.9981484  0.998384   0.998687   0.84839915]
Mean CV accuracy: 0.9684611195135642
Std deviation: 0.06003132519143096


In [56]:
# Grab 5 random test examples and see actual vs predicted
sample_idx = X_test_new.sample(5, random_state=1).index
sample_X = X_test_new.loc[sample_idx]
sample_y_actual = y_test_new.loc[sample_idx]
sample_y_pred = xgb_model_new.predict(sample_X)

comparison = pd.DataFrame({
    'Actual': sample_y_actual.map({0: 'attack', 1: 'normal'}),
    'Predicted': pd.Series(sample_y_pred, index=sample_idx).map({0: 'attack', 1: 'normal'})
})
print(comparison)

        Actual Predicted
42454   normal    normal
112243  normal    normal
32576   attack    attack
88845   attack    attack
98175   normal    normal


In [57]:
from sklearn.model_selection import StratifiedKFold

cv_shuffled = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_shuffled = cross_val_score(xgb_model_new, X_full, y_full, cv=cv_shuffled, scoring='accuracy')

print("Shuffled CV scores:", cv_scores_shuffled)
print("Mean CV accuracy:", cv_scores_shuffled.mean())
print("Std deviation:", cv_scores_shuffled.std())

Shuffled CV scores: [0.99548882 0.99643146 0.99609467 0.99579167 0.99622934]
Mean CV accuracy: 0.9960071917295888
Std deviation: 0.0003323123511728115


In [58]:
import joblib

joblib.dump(xgb_model_new, 'nids_model.pkl')
joblib.dump(list(X_train_new.columns), 'model_columns.pkl')

print("Model and columns saved successfully!")

Model and columns saved successfully!
